In [6]:
!pip install imbalanced-learn


In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, precision_recall_curve
from sklearn.utils.class_weight import compute_class_weight
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

In [8]:
df = pd.read_csv("ProcessedData1.csv")
df.columns = df.columns.str.strip()
df = df.drop(columns=[""], errors="ignore")
df.head()

,enrollment,duration,phase_encoded,condition_encoded,sponsor_type_INDUSTRY,sponsor_type_NIH,sponsor_type_OTHER,gender_MALE,final_status_success,location_United States
0,0.0,0.0,1,5,0,1,0,0,1,0
1,0.0,0.0,1,5,0,0,0,0,0,1
2,-8223.0,3755.0,1,7,0,1,0,1,1,1
3,0.0,0.0,1,1,0,1,0,0,0,1
4,0.0,0.0,1,2,0,1,0,0,0,1


In [9]:
# Feature and target split
X = df.drop("final_status_success", axis=1)
y = df["final_status_success"]

In [10]:
# Compute class weights manually
weights = compute_class_weight(class_weight="balanced", classes=np.unique(y), y=y)
class_weight_dict = {0: weights[0], 1: weights[1]}

In [11]:
# Pipeline: scaling + logistic regression
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(solver="liblinear", max_iter=1000))
])


In [12]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [13]:
pipeline = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('poly', PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(
        solver='liblinear',
        class_weight=class_weight_dict,
        max_iter=1000
    ))
])

In [14]:
# Hyperparameter grid
param_grid = {
    'logreg__C': [0.001, 0.01, 0.1],
    'logreg__penalty': ['l1', 'l2']
}

In [15]:
# Grid search
grid = GridSearchCV(pipeline, param_grid, cv=10, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)


GridSearchCV(cv=10,
             estimator=Pipeline(steps=[('smote', SMOTE(random_state=42)),
                                       ('poly',
                                        PolynomialFeatures(include_bias=False,
                                                           interaction_only=True)),
                                       ('scaler', StandardScaler()),
                                       ('logreg',
                                        LogisticRegression(class_weight={0: np.float64(1.003651732882502),
                                                                         1: np.float64(0.9963747440502165)},
                                                           max_iter=1000,
                                                           solver='liblinear'))]),
             n_jobs=-1,
             param_grid={'logreg__C': [0.001, 0.01, 0.1],
                         'logreg__penalty': ['l1', 'l2']},
             scoring='accuracy')

In [16]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

In [17]:
# Threshold tuning
precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

In [18]:
print("Best Parameters:", grid.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nThreshold tuning values (first 5):")
print("Thresholds:", thresholds[:5])
print("Precision:", precision[:5])
print("Recall:", recall[:5])

Best Parameters: {'logreg__C': 0.001, 'logreg__penalty': 'l2'}
Accuracy: 0.693477070787889

Classification Report:
               precision    recall  f1-score   support

           0       0.62      1.00      0.76     11830
           1       1.00      0.39      0.56     11917

    accuracy                           0.69     23747
   macro avg       0.81      0.69      0.66     23747
weighted avg       0.81      0.69      0.66     23747


Threshold tuning values (first 5):
Thresholds: [0.33872652 0.33902062 0.33931485 0.3396092  0.33990367]
Precision: [0.50183181 0.50485144 0.50843342 0.51208801 0.5150797 ]
Recall: [1.         0.98237812 0.96626668 0.94914828 0.93009986]


In [19]:
from sklearn.metrics import f1_score
from utils import (
    save_pickle,
    compress_model,
    save_classification_report,
    save_roc_curve,
    save_metadata
)

# 1️⃣ Save the trained model pipeline (.pkl and .tar.gz)
save_pickle(pipeline, "models/logistic_regression.pkl")
compress_model("models/logistic_regression.pkl", "models/logistic_regression.tar.gz")
print("✅ Logistic Regression pipeline saved to models/")

# 2️⃣ Save classification report
save_classification_report(y_test, y_pred, "logistic_regression")

# 3️⃣ Save ROC curve
roc_auc = save_roc_curve(y_test, y_proba, "logistic_regression")

# 4️⃣ Save metadata
save_metadata({
    "logistic_regression": {
        "f1_score": f1_score(y_test, y_pred),
        "roc_auc": roc_auc
    }
})


✅ Saved pickle: models/logistic_regression.pkl
📦 Compressed model to: models/logistic_regression.tar.gz
✅ Logistic Regression pipeline saved to models/
📊 Classification report saved: reports/logistic_regression_classification_report.txt
📈 ROC curve saved: reports/logistic_regression_roc_curve.png
📘 Metadata saved: models/metadata.json
